# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print("Record Sets:")
for rs_id in record_sets:
    print(f"- RecordSet @id: {rs_id}")

# List fields within each record set
for rs_id in record_sets:
    rs_meta = next((rs for rs in metadata.to_json().get('recordSet', []) if rs['@id'] == rs_id), None)
    if rs_meta and 'field' in rs_meta:
        print(f"\nFields in RecordSet {rs_id}:")
        for field in rs_meta['field']:
            print(f"- Field @id: {field['@id']} | Name: {field.get('name', 'N/A')}")
    else:
        print(f"No fields found for RecordSet {rs_id}.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If there are record sets, load data from each
dataframes = {}

if record_sets:
    for record_set_id in record_sets:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet: {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
import numpy as np

# Select a record set and numeric field
if record_sets:
    selected_record_set = record_sets[0]
    df = dataframes[selected_record_set]
    
    # Attempt to select a numeric field (example: GAD_7_score@id, PHQ_9_score@id, PSQ_score@id)
    possible_numeric_fields = [col for col in df.columns if 'score' in col.lower() or col.lower() in ['age']]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None

    if numeric_field:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized" ]].head())

        # Group by a categorical field (example: gender@id, marital_status@id)
        possible_group_fields = [col for col in df.columns if 'gender' in col.lower() or 'marital' in col.lower() or 'village' in col.lower()]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for analysis.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic visualization: Distribution of numeric field
if record_sets and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}")
    plt.ylabel("Frequency")
    plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(f"{group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- Successfully loaded metadata and records using `mlcroissant` from the Croissant schema.
- Identified available record sets and explored their fields with `@id` references.
- Extracted data into pandas DataFrames for further analysis.
- Demonstrated filtering and normalization of numeric survey scores (e.g. GAD-7, PHQ-9, PSQ).
- Visualized data distributions and relationships by grouping fields like gender or village.
- This workflow enables reproducible analysis of AI-ready mental health survey datasets.